In [ ]:
!pip install ultralytics

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 71.7 MB/s eta 0:00:00


In [ ]:
!pip install torch torchvision
!pip install git+https://github.com/facebookresearch/segment-anything.git

  Cloning https://github.com/facebookresearch/segment-anything.git to /tmp/pip-req-build-suog1k_3
  Running command git clone --filter=blob:none --quiet https://github.com/facebookresearch/segment-anything.git /tmp/pip-req-build-suog1k_3
  Resolved https://github.com/facebookresearch/segment-anything.git to commit dca509fe793f601edb92606367a655c15ac00fdf
  Preparing metadata (setup.py) ... done
  Created wheel for segment_anything: filename=segment_anything-1.0-py3-none-any.whl size=36592 sha256=0dc5343ec96209720487f4d534fbc4242329cc4e68fd1198baa11e79f80940cf
  Stored in directory: /tmp/pip-ephem-wheel-cache-kv3t4aim/wheels/29/82/ff/04e2be9805a1cb48bec0b85b5a6da6b63f647645750a0e42d4
Successfully built segment_anything


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from ultralytics import YOLO
import cv2
import numpy as np
from segment_anything import SamPredictor, sam_model_registry
from PIL import Image
import matplotlib.pyplot as plt

In [ ]:
!wget -O /content/sam_vit_b_01ec64.pth https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth

--2025-12-11 12:35:17--  https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth
Resolving dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)... 99.84.41.33, 99.84.41.129, 99.84.41.80, ...
Connecting to dl.fbaipublicfiles.com (dl.fbaipublicfiles.com)|99.84.41.33|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 375042383 (358M) [binary/octet-stream]
Saving to: ‘/content/sam_vit_b_01ec64.pth’

/content/sam_vit_b_ 100%[===================>] 357.67M   262MB/s    in 1.4s    

2025-12-11 12:35:18 (262 MB/s) - ‘/content/sam_vit_b_01ec64.pth’ saved [375042383/375042383]



In [ ]:
# =====================================================
# Paths
# =====================================================
images_folder = "/content/drive/My Drive/new grad data/images/train"
output_folder = "/content/drive/My Drive/new grad data/masks/train_masks"
os.makedirs(output_folder, exist_ok=True)

# =====================================================
# Load YOLO Model
# =====================================================
yolo_model = YOLO("/content/drive/My Drive/new grad data/best weights for detection.pt")
yolo_model.to("cuda")

TARGET_CLASS = 0
CONFIDENCE_THRESHOLD = 0.3

# =====================================================
# Load SAM (fast version - vit_b)
# =====================================================
sam_checkpoint = "/content/sam_vit_b_01ec64.pth"
sam = sam_model_registry["vit_b"](checkpoint=sam_checkpoint)
sam.to("cuda")
predictor = SamPredictor(sam)


# =====================================================
# Function to process one image
# =====================================================
def process_and_save_mask(image_path, mask_path):

    image = cv2.imread(image_path)
    if image is None:
        print(f"Cannot read {image_path}")
        return

    # 1) YOLO detection
    det = yolo_model.predict(source=image, verbose=False)[0]

    class_mask = (det.boxes.cls.cpu().numpy() == TARGET_CLASS) & \
                 (det.boxes.conf.cpu().numpy() >= CONFIDENCE_THRESHOLD)

    boxes = det.boxes.xyxy.cpu().numpy()[class_mask]

    if len(boxes) == 0:
        print(f"No valid boxes found → {os.path.basename(image_path)}")
        return

    # 2) SAM segmentation
    predictor.set_image(image)
    H, W = image.shape[:2]
    combined_mask = np.zeros((H, W), dtype=np.uint8)

    for box in boxes:
        x1, y1, x2, y2 = box
        input_box = np.array([x1, y1, x2, y2])

        masks, _, _ = predictor.predict(
            box=input_box,
            multimask_output=False
        )

        combined_mask = np.logical_or(combined_mask, masks[0])

    combined_mask_uint8 = (combined_mask * 255).astype(np.uint8)

    # 3) Save mask
    cv2.imwrite(mask_path, combined_mask_uint8)
    print(f"Saved mask → {mask_path}")


# =====================================================
# LOOP: Process all images in folder
# =====================================================
for img_name in os.listdir(images_folder):

    if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_path = os.path.join(images_folder, img_name)

    name, ext = os.path.splitext(img_name)
    mask_name = f"{name}_mask.png"   # always save as PNG
    mask_path = os.path.join(output_folder, mask_name)

    # -----------------------------------------
    # Skip if mask already exists
    # -----------------------------------------
    if os.path.exists(mask_path):
        print(f"Already processed → {mask_name}")
        continue

    # -----------------------------------------
    # Mask does NOT exist → process it
    # -----------------------------------------
    print(f"Processing new image → {img_name}")
    process_and_save_mask(img_path, mask_path)

print("DONE! All masks processed successfully.")

Streaming output truncated to the last 5000 lines.
Saved mask → /content/drive/My Drive/new grad data/masks/train_masks/52_frame_464_mask.png
Processing new image → 52_frame_519.jpg
Saved mask → /content/drive/My Drive/new grad data/masks/train_masks/52_frame_519_mask.png
Processing new image → 52_frame_507.jpg
Saved mask → /content/drive/My Drive/new grad data/masks/train_masks/52_frame_507_mask.png
Processing new image → 52_frame_422.jpg
Saved mask → /content/drive/My Drive/new grad data/masks/train_masks/52_frame_422_mask.png
Processing new image → 52_frame_420.jpg
Saved mask → /content/drive/My Drive/new grad data/masks/train_masks/52_frame_420_mask.png
Processing new image → 52_frame_473.jpg
Saved mask → /content/drive/My Drive/new grad data/masks/train_masks/52_frame_473_mask.png
Processing new image → 52_frame_699.jpg
Saved mask → /content/drive/My Drive/new grad data/masks/train_masks/52_frame_699_mask.png
Processing new image → 52_frame_677.jpg
Saved mask → /content/drive/My D

In [ ]:
# ===========================================
# Paths (Edit according to your folders)
# ===========================================
images_folder = "/content/drive/My Drive/new grad data/images/val"          # Input images folder
output_folder = "/content/drive/My Drive/new grad data/masks/val_masks"     # Folder to save results
os.makedirs(output_folder, exist_ok=True)

# ===========================================
# Load YOLO detection model
# ===========================================
yolo_model = YOLO("/content/drive/My Drive/new grad data/best weights for detection.pt")
yolo_model.to("cuda")


# Target class to segment (0 = first class)
TARGET_CLASS = 0
CONFIDENCE_THRESHOLD = 0.3  # Filter YOLO boxes below this confidence

# ===========================================
# Load SAM model (smaller & faster version)
# ===========================================
sam_checkpoint = "/content/sam_vit_b_01ec64.pth"  # Make sure you have vit_b checkpoint
sam = sam_model_registry["vit_b"](checkpoint=sam_checkpoint)
sam.to("cuda")
predictor = SamPredictor(sam)

# ===========================================
# Process all images in the folder
# ===========================================
for img_name in os.listdir(images_folder):

    # Skip non-image files
    if not img_name.lower().endswith(('.png', '.jpg', '.jpeg')):
        continue

    img_path = os.path.join(images_folder, img_name)
    image = cv2.imread(img_path)
    if image is None:
        print(f"Cannot read {img_name}, skipping...")
        continue

    print(f"Processing: {img_name}")

    # ---------------------------
    # 1) YOLO Detection
    # ---------------------------
    det = yolo_model.predict(source=image, verbose=False)[0]

    # Filter detections by target class and confidence
    class_mask = (det.boxes.cls.cpu().numpy() == TARGET_CLASS) & (det.boxes.conf.cpu().numpy() >= CONFIDENCE_THRESHOLD)
    boxes = det.boxes.xyxy.cpu().numpy()[class_mask]

    if len(boxes) == 0:
        print(f"No confident class {TARGET_CLASS} found in {img_name}")
        continue

    # ---------------------------
    # 2) Prepare SAM for segmentation
    # ---------------------------
    predictor.set_image(image)
    H, W = image.shape[:2]
    combined_mask = np.zeros((H, W), dtype=np.uint8)

    # ---------------------------
    # 3) Generate mask for each detected box
    # ---------------------------
    for box in boxes:
        x1, y1, x2, y2 = box
        input_box = np.array([x1, y1, x2, y2])

        # Run SAM segmentation
        masks, _, _ = predictor.predict(
            box=input_box,
            multimask_output=False
        )

        # Combine masks into one
        combined_mask = np.logical_or(combined_mask, masks[0])

    # Convert boolean mask to uint8
    combined_mask_uint8 = (combined_mask * 255).astype(np.uint8)

    # ---------------------------
    # 4) Save mask
    # ---------------------------
    name, ext = os.path.splitext(img_name)
    save_name = f"{name}_mask.png"  # Always save as PNG
    save_path = os.path.join(output_folder, save_name)
    cv2.imwrite(save_path, combined_mask_uint8)

    print(f"Saved → {save_path}")

print("✔✔ DONE! All masks generated successfully.")


Processing: 14_frame_736.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_frame_736_mask.png
Processing: 14_frame_862.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_frame_862_mask.png
Processing: 14_frame_763.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_frame_763_mask.png
Processing: 14_frame_74.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_frame_74_mask.png
Processing: 14_frame_731.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_frame_731_mask.png
Processing: 14_frame_764.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_frame_764_mask.png
Processing: 14_frame_794.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_frame_794_mask.png
Processing: 14_frame_902.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_frame_902_mask.png
Processing: 14_frame_916.jpg
Saved → /content/drive/My Drive/new grad data/masks/val_masks/14_fram

In [ ]:
folders = [
    "/content/drive/My Drive/new grad data/images/train",
    "/content/drive/My Drive/new grad data/images/val",
    "/content/drive/My Drive/new grad data/masks/train_masks",
    "/content/drive/My Drive/new grad data/masks/val_masks"
]

def count_images(folder_path):
    count = 0
    for file in os.listdir(folder_path):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            count += 1
    return count

total = 0

for folder in folders:
    c = count_images(folder)
    print(f"Folder: {folder} → {c} images")
    total += c

print("\nTotal images in all folders =", total)


Folder: /content/drive/My Drive/new grad data/images/train → 7095 images
Folder: /content/drive/My Drive/new grad data/images/val → 1729 images
Folder: /content/drive/My Drive/new grad data/masks/train_masks → 7045 images
Folder: /content/drive/My Drive/new grad data/masks/val_masks → 1717 images

Total images in all folders = 17586


In [ ]:
# =============================================
# Paths
# =============================================
train_images = "/content/drive/My Drive/new grad data/images/train"
val_images   = "/content/drive/My Drive/new grad data/images/val"

train_masks  = "/content/drive/My Drive/new grad data/masks/train_masks"
val_masks    = "/content/drive/My Drive/new grad data/masks/val_masks"

os.makedirs(train_masks, exist_ok=True)
os.makedirs(val_masks, exist_ok=True)

# ======================================================
# Function: create black mask with same size as image
# ======================================================
def create_black_mask(image_path, save_mask_path):

    img = cv2.imread(image_path)
    if img is None:
        print(f"Error reading {image_path}")
        return

    H, W = img.shape[:2]
    black_mask = np.zeros((H, W), dtype=np.uint8)

    cv2.imwrite(save_mask_path, black_mask)
    print(f"Created black mask → {save_mask_path}")


# ======================================================
# Function: ensure every image has mask
# ======================================================
def ensure_masks(images_folder, masks_folder):

    images = sorted([f for f in os.listdir(images_folder)
                     if f.lower().endswith(('.png','.jpg','.jpeg'))])

    masks = sorted([f for f in os.listdir(masks_folder)
                    if f.lower().endswith(('.png','.jpg','.jpeg','.bmp'))])

    print(f"\nChecking folder:\nImages → {images_folder}\nMasks → {masks_folder}\n")

    img_set = set(images)
    mask_set = set([m.replace("_mask","") for m in masks])   # remove _mask for matching

    for img_name in img_set:

        name, ext = os.path.splitext(img_name)
        mask_name = f"{name}_mask.png"
        mask_path = os.path.join(masks_folder, mask_name)

        if not os.path.exists(mask_path):
            # Mask does NOT exist → create black mask
            create_black_mask(os.path.join(images_folder, img_name), mask_path)

    print("\n Checking done!")


# ======================================================
# Run for train + val
# ======================================================
ensure_masks(train_images, train_masks)
ensure_masks(val_images, val_masks)

print("\n All missing masks have been created!")



Checking folder:
Images → /content/drive/My Drive/new grad data/images/train
Masks → /content/drive/My Drive/new grad data/masks/train_masks

Created black mask → /content/drive/My Drive/new grad data/masks/train_masks/23_frame_1052_mask.png
Created black mask → /content/drive/My Drive/new grad data/masks/train_masks/23_frame_976_mask.png
Created black mask → /content/drive/My Drive/new grad data/masks/train_masks/23_frame_1140_mask.png
Created black mask → /content/drive/My Drive/new grad data/masks/train_masks/23_frame_1191_mask.png
Created black mask → /content/drive/My Drive/new grad data/masks/train_masks/23_frame_1173_mask.png
Created black mask → /content/drive/My Drive/new grad data/masks/train_masks/23_frame_1206_mask.png
Created black mask → /content/drive/My Drive/new grad data/masks/train_masks/23_frame_1066_mask.png
Created black mask → /content/drive/My Drive/new grad data/masks/train_masks/23_frame_1067_mask.png
Created black mask → /content/drive/My Drive/new grad data

In [ ]:
folders = [
    "/content/drive/My Drive/new grad data/images/train",
    "/content/drive/My Drive/new grad data/images/val",
    "/content/drive/My Drive/new grad data/masks/train_masks",
    "/content/drive/My Drive/new grad data/masks/val_masks"
]

def count_images(folder_path):
    count = 0
    for file in os.listdir(folder_path):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            count += 1
    return count

total = 0

for folder in folders:
    c = count_images(folder)
    print(f"Folder: {folder} → {c} images")
    total += c

print("\nTotal images in all folders =", total)


Folder: /content/drive/My Drive/new grad data/images/train → 7095 images
Folder: /content/drive/My Drive/new grad data/images/val → 1729 images
Folder: /content/drive/My Drive/new grad data/masks/train_masks → 7103 images
Folder: /content/drive/My Drive/new grad data/masks/val_masks → 1729 images

Total images in all folders = 17656


In [ ]:
# ============================================
# Paths
# ============================================
train_images = "/content/drive/My Drive/new grad data/images/train"
val_images   = "/content/drive/My Drive/new grad data/images/val"

train_masks  = "/content/drive/My Drive/new grad data/masks/train_masks"
val_masks    = "/content/drive/My Drive/new grad data/masks/val_masks"


def clean_extra_masks(images_folder, masks_folder):
    print(f"\nChecking folder:\nImages → {images_folder}\nMasks → {masks_folder}")

    # ----------- Get image names (without extension)
    image_files = [f for f in os.listdir(images_folder)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    image_names = set(os.path.splitext(f)[0] for f in image_files)

    # ----------- Get mask names (remove _mask suffix)
    mask_files = [f for f in os.listdir(masks_folder)
                  if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    mask_names = set(os.path.splitext(f)[0].replace("_mask", "") for f in mask_files)

    # ----------- Identify extra masks (masks without images)
    extra_masks = [f for f in mask_files
                   if os.path.splitext(f)[0].replace("_mask", "") not in image_names]

    # ----------- Identify missing masks
    missing_masks = [img for img in image_names
                     if f"{img}_mask.png" not in mask_files and f"{img}_mask.jpg" not in mask_files]

    # ----------- Print summary
    print(f"\nTotal images: {len(image_files)}")
    print(f"Total masks: {len(mask_files)}")
    print(f"Extra masks (will be deleted): {len(extra_masks)}")
    print(f"Missing masks: {len(missing_masks)}")

    # ----------- Delete extra masks
    for mask in extra_masks:
        mask_path = os.path.join(masks_folder, mask)
        os.remove(mask_path)
        print(f"Deleted extra mask → {mask}")

    print("\nCleanup completed.\n")


# =====================================================
# Run cleaning for both train and val
# =====================================================
clean_extra_masks(train_images, train_masks)
clean_extra_masks(val_images, val_masks)



Checking folder:
Images → /content/drive/My Drive/new grad data/images/train
Masks → /content/drive/My Drive/new grad data/masks/train_masks

Total images: 7095
Total masks: 7103
Extra masks (will be deleted): 8
Missing masks: 0
Deleted extra mask → 12_frame_1428_mask (1).png
Deleted extra mask → 12_frame_1343_mask (1).png
Deleted extra mask → 12_frame_329_mask (1).png
Deleted extra mask → 12_frame_541_mask (1).png
Deleted extra mask → 60_frame_614_mask (1).png
Deleted extra mask → 82_frame_873_mask (1).png
Deleted extra mask → 13_frame_582_mask (1).png
Deleted extra mask → 47_frame_546_mask (1).png

Cleanup completed.


Checking folder:
Images → /content/drive/My Drive/new grad data/images/val
Masks → /content/drive/My Drive/new grad data/masks/val_masks

Total images: 1729
Total masks: 1729
Extra masks (will be deleted): 0
Missing masks: 0

Cleanup completed.



In [ ]:
folders = [
    "/content/drive/My Drive/new grad data/images/train",
    "/content/drive/My Drive/new grad data/images/val",
    "/content/drive/My Drive/new grad data/masks/train_masks",
    "/content/drive/My Drive/new grad data/masks/val_masks"
]

def count_images(folder_path):
    count = 0
    for file in os.listdir(folder_path):
        if file.lower().endswith((".png", ".jpg", ".jpeg")):
            count += 1
    return count

total = 0

for folder in folders:
    c = count_images(folder)
    print(f"Folder: {folder} → {c} images")
    total += c

print("\nTotal images in all folders =", total)


Folder: /content/drive/My Drive/new grad data/images/train → 7095 images
Folder: /content/drive/My Drive/new grad data/images/val → 1729 images
Folder: /content/drive/My Drive/new grad data/masks/train_masks → 7095 images
Folder: /content/drive/My Drive/new grad data/masks/val_masks → 1729 images

Total images in all folders = 17648


In [ ]:
import cv2
mask = cv2.imread("/content/drive/My Drive/new grad data/masks/train_masks/11_frame_8_mask.png", cv2.IMREAD_UNCHANGED)
print(mask.shape)

(480, 640)


In [ ]:
import cv2
import numpy as np
mask = cv2.imread("/content/drive/My Drive/new grad data/masks/train_masks/11_frame_8_mask.png", cv2.IMREAD_GRAYSCALE)
print(np.unique(mask))


[  0 255]


In [ ]:
import os

img_folder = "/content/drive/My Drive/new grad data/images/train"
mask_folder = "/content/drive/My Drive/new grad data/masks/train_masks"

imgs = sorted([f for f in os.listdir(img_folder) if f.endswith(('.jpg', '.png'))])
masks = sorted([f for f in os.listdir(mask_folder) if f.endswith('.png')])

for img in imgs:
    mask_name = os.path.splitext(img)[0] + "_mask.png"
    if mask_name not in masks:
        print("Missing mask for:", img)


In [ ]:
import os
import cv2
import numpy as np

# =====================================================
# Folders
# =====================================================
folders = {
    "train": {
        "images": "/content/drive/My Drive/new grad data/images/train",
        "masks":  "/content/drive/My Drive/new grad data/masks/train_masks"
    },
    "val": {
        "images": "/content/drive/My Drive/new grad data/images/val",
        "masks":  "/content/drive/My Drive/new grad data/masks/val_masks"
    }
}

# =====================================================
# Function to fix mask
# =====================================================
def fix_mask(mask_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        print(f"Cannot read mask {mask_path}")
        return False

    # Convert any non-zero pixel to 255
    mask[mask > 0] = 255
    cv2.imwrite(mask_path, mask)
    return True

# =====================================================
# Main check
# =====================================================
for split in ["train", "val"]:
    img_folder = folders[split]["images"]
    mask_folder = folders[split]["masks"]

    img_files = sorted([f for f in os.listdir(img_folder) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
    mask_files = sorted([f for f in os.listdir(mask_folder) if f.lower().endswith('.png')])

    # Check for missing masks
    for img_name in img_files:
        mask_name = os.path.splitext(img_name)[0] + "_mask.png"
        mask_path = os.path.join(mask_folder, mask_name)
        if not os.path.exists(mask_path):
            # create black mask if missing
            img_path = os.path.join(img_folder, img_name)
            img = cv2.imread(img_path)
            if img is not None:
                H, W = img.shape[:2]
                black_mask = np.zeros((H, W), dtype=np.uint8)
                cv2.imwrite(mask_path, black_mask)
                print(f"Created black mask for missing: {mask_name}")
        else:
            # fix mask values
            fix_mask(mask_path)

    # Optional: Remove masks without corresponding images
    for mask_name in mask_files:
        img_name = mask_name.replace("_mask.png", "")
        img_extensions = ['.jpg', '.jpeg', '.png']
        if not any(os.path.exists(os.path.join(img_folder, img_name + ext)) for ext in img_extensions):
            os.remove(os.path.join(mask_folder, mask_name))
            print(f"Removed mask without image: {mask_name}")

print("All masks checked and fixed.")


All masks checked and fixed.


In [ ]:
pip install -U ultralytics

In [ ]:
base_folder = "/content/drive/My Drive/new grad data/segmented_labels"
os.makedirs(os.path.join(base_folder, "train_masks_txt"), exist_ok=True)
os.makedirs(os.path.join(base_folder, "val_masks_txt"), exist_ok=True)

In [ ]:
import os
import cv2
import numpy as np

# ===============================
# Paths
# ===============================
train_masks_folder = "/content/drive/My Drive/new grad data/masks/train_masks"
val_masks_folder   = "/content/drive/My Drive/new grad data/masks/val_masks"

train_txt_folder = "/content/drive/My Drive/new grad data/segmented_labels/train_masks_txt"
val_txt_folder   = "/content/drive/My Drive/new grad data/segmented_labels/val_masks_txt"

# Create folders if they don't exist
os.makedirs(train_txt_folder, exist_ok=True)
os.makedirs(val_txt_folder, exist_ok=True)

# ===============================
# Function to convert a mask to YOLO segmentation TXT
# ===============================
def mask_to_yolo_seg(mask_path, txt_path):
    mask = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
    if mask is None:
        print(f"Cannot read mask: {mask_path}")
        return

    H, W = mask.shape
    # Binarize mask (0 background, 1 object)
    bin_mask = (mask > 0).astype(np.uint8)

    # Find contours
    contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    lines = []
    for cnt in contours:
        if len(cnt) < 1:
            continue
        # Flatten contour points
        cnt = cnt.reshape(-1, 2)
        x = cnt[:, 0] / W  # normalize x
        y = cnt[:, 1] / H  # normalize y
        # YOLO segmentation format: class_index x1 y1 x2 y2 ...
        coords = ' '.join([f"{xi:.6f} {yi:.6f}" for xi, yi in zip(x, y)])
        lines.append(f"0 {coords}")  # class_index = 0

    # Write TXT (even if empty)
    with open(txt_path, "w") as f:
        f.write("\n".join(lines))

# ===============================
# Process all masks
# ===============================
def process_masks(masks_folder, txt_folder):
    for mask_name in os.listdir(masks_folder):
        if not mask_name.lower().endswith(('.png', '.jpg', '.jpeg')):
            continue

        mask_path = os.path.join(masks_folder, mask_name)
        name, _ = os.path.splitext(mask_name)
        txt_path = os.path.join(txt_folder, f"{name}.txt")

        mask_to_yolo_seg(mask_path, txt_path)

# Process train and val
process_masks(train_masks_folder, train_txt_folder)
process_masks(val_masks_folder, val_txt_folder)

print("DONE! All masks converted to YOLO segmentation format.")


DONE! All masks converted to YOLO segmentation format.


In [ ]:
import os

# Paths to your mask txt folders
train_masks_folder = "/content/drive/My Drive/new grad data/segmented_labels/train_masks_txt"
val_masks_folder   = "/content/drive/My Drive/new grad data/segmented_labels/val_masks_txt"

# Count txt files
train_count = len([f for f in os.listdir(train_masks_folder) if f.lower().endswith('.txt')])
val_count   = len([f for f in os.listdir(val_masks_folder) if f.lower().endswith('.txt')])

print(f"Number of train mask txt files: {train_count}")
print(f"Number of val mask txt files: {val_count}")


Number of train mask txt files: 7095
Number of val mask txt files: 1729


In [ ]:
import os

images_folder = "/content/drive/My Drive/new grad data/images"
masks_folder = "/content/drive/My Drive/new grad data/masks"

images = sorted([f for f in os.listdir(images_folder) if f.lower().endswith((".jpg", ".png", ".jpeg"))])
masks = sorted([f for f in os.listdir(masks_folder) if f.lower().endswith((".txt", ".png", ".jpg"))])

missing_masks = []

for img in images:
    base = os.path.splitext(img)[0]
    expected_mask = base + "_mask"

    # check if mask name appears in any mask file
    found = any(expected_mask in m for m in masks)

    if not found:
        missing_masks.append(img)

print("num og images without masks", len(missing_masks))
print("missing images: ")
for m in missing_masks:
    print(m)


num og images without masks 0
missing images: 


In [ ]:
import os

# Paths to your train/val mask folders
train_masks = "/content/drive/My Drive/new grad data/labels/train"
val_masks   = "/content/drive/My Drive/new grad data/labels/val"

def rename_masks(folder):
    for filename in os.listdir(folder):
        if filename.endswith(".txt") and "_mask" in filename:
            old_path = os.path.join(folder, filename)
            new_name = filename.replace("_mask", "")   # remove _mask
            new_path = os.path.join(folder, new_name)

            # Rename the file
            os.rename(old_path, new_path)
            print(f"Renamed: {filename} → {new_name}")

print("Renaming train masks...")
rename_masks(train_masks)

print("\nRenaming val masks...")
rename_masks(val_masks)


Streaming output truncated to the last 5000 lines.
Renamed: 30_frame_1411_mask.txt → 30_frame_1411.txt
Renamed: 30_frame_1404_mask.txt → 30_frame_1404.txt
Renamed: 30_frame_1414_mask.txt → 30_frame_1414.txt
Renamed: 30_frame_143_mask.txt → 30_frame_143.txt
Renamed: 30_frame_1433_mask.txt → 30_frame_1433.txt
Renamed: 30_frame_1424_mask.txt → 30_frame_1424.txt
Renamed: 30_frame_1442_mask.txt → 30_frame_1442.txt
Renamed: 30_frame_1446_mask.txt → 30_frame_1446.txt
Renamed: 30_frame_1456_mask.txt → 30_frame_1456.txt
Renamed: 30_frame_146_mask.txt → 30_frame_146.txt
Renamed: 30_frame_1463_mask.txt → 30_frame_1463.txt
Renamed: 30_frame_1465_mask.txt → 30_frame_1465.txt
Renamed: 30_frame_1437_mask.txt → 30_frame_1437.txt
Renamed: 30_frame_153_mask.txt → 30_frame_153.txt
Renamed: 30_frame_1469_mask.txt → 30_frame_1469.txt
Renamed: 30_frame_154_mask.txt → 30_frame_154.txt
Renamed: 30_frame_157_mask.txt → 30_frame_157.txt
Renamed: 30_frame_160_mask.txt → 30_frame_160.txt
Renamed: 30_frame_162_mas

In [ ]:
import os

# Paths
images_folder = "/content/drive/My Drive/new grad data/images/train"
masks_folder  = "/content/drive/My Drive/new grad data/labels/train"

# Get image base names (without extension)
image_names = {os.path.splitext(f)[0] for f in os.listdir(images_folder)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))}

# Get mask base names
mask_names = {os.path.splitext(f)[0] for f in os.listdir(masks_folder)
              if f.lower().endswith(".txt")}

# Find missing masks
missing_masks = image_names - mask_names
# Find masks with no images
extra_masks = mask_names - image_names

print("✔ Total images:", len(image_names))
print("✔ Total masks:", len(mask_names))

print("\n Images WITHOUT masks:")
for name in sorted(missing_masks):
    print(name)

print("\n Masks with NO matching images:")
for name in sorted(extra_masks):
    print(name)


✔ Total images: 7095
✔ Total masks: 7095

 Images WITHOUT masks:

 Masks with NO matching images:


In [ ]:
import os

# Paths
images_folder = "/content/drive/My Drive/new grad data/images/val"
masks_folder  = "/content/drive/My Drive/new grad data/labels/val"

# Get image base names (without extension)
image_names = {os.path.splitext(f)[0] for f in os.listdir(images_folder)
               if f.lower().endswith((".jpg", ".jpeg", ".png"))}

# Get mask base names
mask_names = {os.path.splitext(f)[0] for f in os.listdir(masks_folder)
              if f.lower().endswith(".txt")}

# Find missing masks
missing_masks = image_names - mask_names
# Find masks with no images
extra_masks = mask_names - image_names

print("✔ Total images:", len(image_names))
print("✔ Total masks:", len(mask_names))

print("\n Images WITHOUT masks:")
for name in sorted(missing_masks):
    print(name)

print("\n Masks with NO matching images:")
for name in sorted(extra_masks):
    print(name)


✔ Total images: 1729
✔ Total masks: 1729

 Images WITHOUT masks:

 Masks with NO matching images:


In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

In [ ]:
from ultralytics import YOLO

# Load segmentation model (pretrained)
model = YOLO("yolov8n-seg.pt")

# Train
results = model.train(
    data="/content/drive/My Drive/new grad data/sperm_dataset_seg.yaml",
    imgsz=640,
    epochs=15,
    batch=16,
    patience=5,
    optimizer="Adam",
    lr0=0.001,
    lrf=0.01,
    weight_decay=0.0005,
    dropout=0.0,

    # Augmentations
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    fliplr=0.5,
    mosaic=0.0,
    mixup=0.0,

    # Accelerate training
    device=0,
    workers=4,

    # Saving configs
    project="/content/drive/My Drive/new grad data/segmentation_results",
    name="yolov8_sperm_seg",
    save=True,
    save_period=2,
    exist_ok=True
)


Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/My Drive/new grad data/sperm_dataset_seg.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=15, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=0.0, multi_scale=False, name=yolov8_sperm_seg, nbs=64, nms=False, opset=None, optimize=False, optimizer=Adam, overlap_mask=True, patie

In [ ]:
# model evaluation
from ultralytics import YOLO
import pandas as pd

model_path = "/content/drive/My Drive/new grad data/segmentation_results/yolov8_sperm_seg/weights/best.pt"
model = YOLO(model_path)


In [ ]:
metrics = model.val(
    data="/content/drive/My Drive/new grad data/sperm_dataset.yaml",
    split="val",
    save_json=True,
    imgsz=640
)

Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 34.0±26.9 MB/s, size: 60.7 KB)
val: Scanning /content/drive/My Drive/new grad data/labels/val.cache... 1729 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1729/1729 361.9Kit/s 0.0s
requirements: Ultralytics requirement ['faster-coco-eval>=1.6.7'] not found, attempting AutoUpdate...
Using Python 3.12.12 environment at: /usr
Resolved 2 packages in 138ms
Prepared 1 package in 24ms
Installed 1 package in 5ms
 + faster-coco-eval==1.7.0

requirements: AutoUpdate success ✅ 1.1s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 109/109 1.4it/s 1:20
                   all       1729      

In [ ]:
box_results = {
    "Box_Precision": metrics.box.p,
    "Box_Recall": metrics.box.r,
    "Box_mAP50": metrics.box.map50,
    "Box_mAP50_95": metrics.box.map
}
mask_results = {
    "Mask_Precision": metrics.seg.p,
    "Mask_Recall": metrics.seg.r,
    "Mask_mAP50": metrics.seg.map50,
    "Mask_mAP50_95": metrics.seg.map
}
results = {**box_results, **mask_results}
df = pd.DataFrame([results])
df


,Box_Precision,Box_Recall,Box_mAP50,Box_mAP50_95,Mask_Precision,Mask_Recall,Mask_mAP50,Mask_mAP50_95
0,[0.9695341554015603],[0.9330492124838372],0.961644,0.679559,[0.9700586705271114],[0.9324869849799307],0.960988,0.521446


In [ ]:
csv_path = "/content/drive/My Drive/new grad data/sperm_segmentation_model_results.csv"
df.to_csv(csv_path, index=False)

print(f"Evaluation results saved to: {csv_path}")
df


Evaluation results saved to: /content/drive/My Drive/new grad data/sperm_segmentation_model_results.csv


,Box_Precision,Box_Recall,Box_mAP50,Box_mAP50_95,Mask_Precision,Mask_Recall,Mask_mAP50,Mask_mAP50_95
0,[0.9695341554015603],[0.9330492124838372],0.961644,0.679559,[0.9700586705271114],[0.9324869849799307],0.960988,0.521446


In [ ]:
# compare
from ultralytics import YOLO

def evaluate_model(model_path, data_yaml):
    model = YOLO(model_path)
    metrics = model.val(
        data=data_yaml,
        split="val",
        imgsz=640,
        save_json=False
    )

    return {
        "Box_mAP50": metrics.box.map50,
        "Box_mAP50_95": metrics.box.map,
        "Mask_mAP50": metrics.seg.map50,
        "Mask_mAP50_95": metrics.seg.map
    }


In [ ]:
model_A_path = "/content/drive/My Drive/new grad data/segmentation_results/yolov8_sperm_seg/weights/best.pt"
model_B_path = "/content/seg.pt"

data_yaml = "/content/drive/My Drive/new grad data/sperm_dataset_seg.yaml"

results_A = evaluate_model(model_A_path, data_yaml)
results_B = evaluate_model(model_B_path, data_yaml)

print("Model A Results:", results_A)
print("Model B Results:", results_B)


Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,259 parameters, 0 gradients, 11.3 GFLOPs
val: Fast image access ✅ (ping: 0.3±0.1 ms, read: 24.5±10.7 MB/s, size: 53.2 KB)
val: Scanning /content/drive/My Drive/new grad data/labels/val.cache... 1729 images, 12 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 1729/1729 1.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 109/109 2.2it/s 49.4s
                   all       1729      35335       0.97      0.933      0.962       0.68      0.936      0.896      0.922      0.452
Speed: 1.6ms preprocess, 4.3ms inference, 0.0ms loss, 3.4ms postprocess per image
Results saved to /content/runs/segment/val2
Ultralytics 8.3.237 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
YOLOv8n-seg summary (fused): 85 layers, 3,258,649 parameters, 0 gradients, 11.3

In [ ]:
import pandas as pd

df = pd.DataFrame([
    {"Model": "YOLOv8n-seg", **results_A},
    {"Model": "YOLOv11-seg", **results_B}
])

df


,Model,Box_mAP50,Box_mAP50_95,Mask_mAP50,Mask_mAP50_95
0,YOLOv8n-seg,0.961644,0.679559,0.92163,0.451648
1,YOLOv11-seg,0.000000,0.000000,0.00000,0.000000
